# Model save and local reload

This notebook directly calls model/tokenizer `save_pretrained()`, lists the saved structure, and reloads with `local_files_only=True`.

In [ ]:
%pip install -q "transformers==4.57.6" "torch>=2.7,<3"

In [ ]:
from pathlib import Path

from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "sshleifer/tiny-gpt2"
save_dir = Path("saved_tiny_model")
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print(
    "\n".join(
        str(path.relative_to(save_dir))
        for path in sorted(save_dir.rglob("*"))
        if path.is_file()
    )
)

In [ ]:
try:
    reloaded_model = AutoModelForCausalLM.from_pretrained(
        save_dir, local_files_only=True
    )
    reloaded_tokenizer = AutoTokenizer.from_pretrained(save_dir, local_files_only=True)
    inputs = reloaded_tokenizer("Local reload works because", return_tensors="pt")
    output = reloaded_model.generate(**inputs, max_new_tokens=12, do_sample=False)
    print(reloaded_tokenizer.decode(output[0], skip_special_tokens=True))
except Exception as exc:  # noqa: BLE001 - notebook reports local reload failures
    print(f"Local reload failed: {type(exc).__name__}: {exc}")

## Expected output and directory

The directory normally includes `config.json`, model weights, tokenizer configuration, vocabulary files, and generation configuration. BitsAndBytes models may require adapter-only saving or dequantization; verify support for the selected architecture before treating a quantized checkpoint as portable.